# Part 5: scrape questions

Numbers only. Answers go in DESIGN.md Part 5.

In [ ]:
import json
import re
import statistics
from pathlib import Path

# Repo root: works whether the notebook runs from notebook/ or from the repo root.
ROOT = Path.cwd() if (Path.cwd() / "metrics").exists() else Path.cwd().parent
RUN_DIR = ROOT / "metrics" / "rehearsal"   # change to the run you want to analyse
WARMUP_DIR = ROOT / "metrics"


def load_jsonl(pattern, folder=None):
    rows = []
    for p in sorted((folder or RUN_DIR).glob(pattern)):
        rows += [json.loads(line) for line in p.read_text().splitlines() if line.strip()]
    if not rows:
        print(f"missing: no rows for {pattern} in {folder or RUN_DIR}")
    return rows


def pct(values, p):
    values = sorted(v for v in values if v is not None)
    return values[min(len(values) - 1, int(round(p / 100 * (len(values) - 1))))] if values else None


def fmt(x, unit="", nd=1):
    return "n/a" if x is None else f"{x:.{nd}f}{unit}"


scrape = load_jsonl("scrape_*.jsonl")
traffic = load_jsonl("traffic_*.jsonl")
ok_scrape = [r for r in scrape if r.get("ok")]
workers = sorted({r["worker"] for r in ok_scrape})


def mix_windows():
    w = {}
    for r in traffic:
        lo, hi = r["ts"], r["ts"] + (r["total_s"] or 0)
        a, b = w.get(r["mix"], (lo, hi))
        w[r["mix"]] = (min(a, lo), max(b, hi))
    return dict(sorted(w.items(), key=lambda kv: kv[1][0]))


WINDOWS = mix_windows()


def rows_in(worker, mix):
    lo, hi = WINDOWS[mix]
    return [r for r in ok_scrape if r["worker"] == worker and lo <= r["ts"] <= hi]


def delta(worker, mix, field):
    """Counter change across a mix window: last row at/before start -> first row at/after end."""
    lo, hi = WINDOWS[mix]
    rows = [r for r in ok_scrape if r["worker"] == worker and r.get(field) is not None]
    before = [r for r in rows if r["ts"] <= lo] or [r for r in rows if lo < r["ts"] < hi][:1]
    after = [r for r in rows if r["ts"] >= hi] or [r for r in rows if lo < r["ts"] < hi][-1:]
    if not before or not after:
        return None
    return after[0][field] - before[-1][field]


print(f"RUN_DIR = {RUN_DIR}")
print(f"{len(scrape)} scrape rows, {len(traffic)} traffic rows, workers {workers}, mixes {list(WINDOWS)}")


## 1. Who sits in **your** queue vs vLLM's **waiting** queue?

In [ ]:
gateway = load_jsonl("gateway_queue*.jsonl")  # written by the gateway once control/ has a queue
print(f"gateway queue rows: {len(gateway)}")
for mix in WINDOWS:
    for w in workers:
        rows = rows_in(w, mix)
        waiting = [r["waiting"] for r in rows]
        dq, dn = delta(w, mix, "queue_time_sum"), delta(w, mix, "queue_time_count")
        mean_q = dq / dn * 1000 if dq is not None and dn else None
        print(f"{mix:15} {w}: engine waiting mean {fmt(statistics.mean(waiting) if waiting else None)}, "
              f"max {fmt(max(waiting) if waiting else None, nd=0)}; engine queue time mean {fmt(mean_q, ' ms')}")
for mix in WINDOWS:
    users = {r.get("users", r.get("concurrency")) for r in traffic if r["mix"] == mix}
    print(f"{mix:15} client in-flight cap (closed-loop users): {sorted(users)}")


## 2. Waiting / running / swapped (or preempted)?

In [ ]:
for mix in WINDOWS:
    for w in workers:
        rows = rows_in(w, mix)
        run = [r["running"] for r in rows]
        wait = [r["waiting"] for r in rows]
        print(f"{mix:15} {w}: running mean {fmt(statistics.mean(run) if run else None)} max {fmt(max(run) if run else None, nd=0)} | "
              f"waiting mean {fmt(statistics.mean(wait) if wait else None)} max {fmt(max(wait) if wait else None, nd=0)} | "
              f"preemptions {fmt(delta(w, mix, 'preemptions'), nd=0)}")


## 3. `orch_replica_queue_depth` (or your equivalent): which pod, under which mix?

In [ ]:
# equivalent for now: engine waiting + running per worker (no gateway queue yet)
for mix in WINDOWS:
    for w in workers:
        depth = [r["waiting"] + r["running"] for r in rows_in(w, mix)]
        print(f"{mix:15} {w}: depth mean {fmt(statistics.mean(depth) if depth else None)}, "
              f"p95 {fmt(pct(depth, 95), nd=0)}, max {fmt(max(depth) if depth else None, nd=0)}")
for mix in WINDOWS:
    targets = {}
    for r in traffic:
        if r["mix"] == mix:
            targets[r["target"]] = targets.get(r["target"], 0) + 1
    print(f"{mix:15} requests per target: {targets}")


## 4. If a 32k RAG retrieve and a short agent decode are both ready, who goes first: you or the engine?

In [ ]:
import sys
sys.path.insert(0, str(ROOT / "cluster"))
src = (ROOT / "cluster" / "modal_vllm.py").read_text()
print("scheduling flags in ENGINE_ARGS:", [a for a in re.findall(r'"(--[a-z-]+)"', src) if "sched" in a] or "none (vLLM default policy)")
print("--max-num-batched-tokens:", re.search(r'"--max-num-batched-tokens", "(\d+)"', src).group(1))
mixed = [r for r in traffic if r["mix"] == "mixed" and r["status"] == 200]
for tenant in ("interactive", "batch"):
    t = [r["ttft_s"] * 1000 for r in mixed if r["tenant"] == tenant and r["ttft_s"]]
    p = [r["prompt_tokens"] for r in mixed if r["tenant"] == tenant and r["prompt_tokens"]]
    budget = sorted({r.get("max_tokens") for r in mixed if r["tenant"] == tenant})
    print(f"mixed / {tenant:11}: n {len(t)}, max_tokens {budget}, prompt tokens median {fmt(statistics.median(p) if p else None, nd=0)}, "
          f"TTFT p50 {fmt(pct(t, 50), ' ms', 0)}, p95 {fmt(pct(t, 95), ' ms', 0)}, p99 {fmt(pct(t, 99), ' ms', 0)}")


## 5. PagedAttention **packs** KV (block table). Prefix / radix cache **reuses** KV. Which one saved memory on your shared-prefix mix?

In [ ]:
for mix in WINDOWS:
    for w in workers:
        dq, dh = delta(w, mix, "prefix_queries"), delta(w, mix, "prefix_hits")
        kv = [r["kv_usage"] * 100 for r in rows_in(w, mix) if r.get("kv_usage") is not None]
        rate = 100 * dh / dq if dq else None
        print(f"{mix:15} {w}: prefix queries {fmt(dq, nd=0)} tokens, hits {fmt(dh, nd=0)} tokens, hit rate {fmt(rate, ' %')} | "
              f"KV usage mean {fmt(statistics.mean(kv) if kv else None, ' %')}, max {fmt(max(kv) if kv else None, ' %')}")


## 6. Chunked prefill and continuous batching: which engine flags (`max-num-batched-tokens`, `max-num-seqs`) did you use and why?

In [ ]:
block = re.search(r"ENGINE_ARGS = \[(.*?)\n\]", src, re.S).group(1)
for line in block.splitlines():
    if line.strip():
        print(line.rstrip())
max_seqs = int(re.search(r'"--max-num-seqs", "(\d+)"', src).group(1))
for mix in WINDOWS:
    for w in workers:
        run = [r["running"] for r in rows_in(w, mix)]
        print(f"{mix:15} {w}: max running {fmt(max(run) if run else None, nd=0)} of {max_seqs} slots")


## 7. KV full after admit: do you shed at the door, or does the engine preempt?

In [ ]:
for mix in WINDOWS:
    for w in workers:
        kv = [r["kv_usage"] * 100 for r in rows_in(w, mix) if r.get("kv_usage") is not None]
        print(f"{mix:15} {w}: KV usage max {fmt(max(kv) if kv else None, ' %')}, preemptions {fmt(delta(w, mix, 'preemptions'), nd=0)}")
for mix in WINDOWS:
    codes = {}
    for r in traffic:
        if r["mix"] == mix:
            codes[str(r["status"])] = codes.get(str(r["status"]), 0) + 1
    print(f"{mix:15} client status codes: {codes}")


## 8. Client gone (`aborted`): who frees the KV? And how?

In [ ]:
for w in workers:
    rows = [r for r in ok_scrape if r["worker"] == w and r.get("request_success_by_finished_reason")]
    if len(rows) >= 2:
        first, last = rows[0]["request_success_by_finished_reason"], rows[-1]["request_success_by_finished_reason"]
        print(f"{w}: finished_reason change over the run: { {k: last.get(k, 0) - first.get(k, 0) for k in last} }")
    else:
        print(f"{w}: missing finished_reason rows")
errors = [r for r in traffic if r["status"] != 200]
print(f"client-side failed requests: {len(errors)}")
for r in errors[:5]:
    print(f"  {r['mix']} {r['status']} {r['error']}")


## 9. After a worker returns: slam it at 100 %, or ramp while p99 holds?

In [ ]:
for path in sorted(WARMUP_DIR.glob("warmup_*.json")) + sorted(RUN_DIR.glob("warmup_*.json")):
    d = json.loads(path.read_text())
    eng = d.get("engine_ttft_s") or []
    shown = path.relative_to(ROOT) if path.is_relative_to(ROOT) else path
    print(f"{shown}: engine TTFT request 1 {fmt(eng[0] * 1000 if eng else None, ' ms')}, "
          f"request 5 {fmt(eng[-1] * 1000 if eng else None, ' ms')}")
for mix in WINDOWS:
    rows = sorted((r for r in traffic if r["mix"] == mix and r["status"] == 200 and r["ttft_s"]), key=lambda r: r["ts"])
    if not rows:
        continue
    t0 = rows[0]["ts"]
    first = [r["ttft_s"] * 1000 for r in rows if r["ts"] - t0 < 60]
    rest = [r["ttft_s"] * 1000 for r in rows if r["ts"] - t0 >= 60]
    print(f"{mix:15}: TTFT p99 first 60 s {fmt(pct(first, 99), ' ms', 0)} (n {len(first)}), "
          f"after {fmt(pct(rest, 99), ' ms', 0)} (n {len(rest)})")
